For each file, find its person id and split it into 5 second chunks

In [13]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
from types import SimpleNamespace

# ---- Paths ----
kalman_filter_data_path = '../data/kaggle-drdataboston/attempt_3/updated_data_kalman_filtered_3'
matrix_path = '../data/kaggle-drdataboston/matrix.csv'
out_path = '../data/kaggle-drdataboston/attempt_5'

# ---- Parameters ----
window_ms = 5000
interp_pts = 500

# ---- Load person-session mapping ----
matrix = pd.read_csv(matrix_path)
session_cols = [
    "subject_left_waist_session1", "subject_right_pocket_session1",
    "subject_left_waist_session2", "subject_right_pocket_session2"
]

file_to_person = {}
for _, row in matrix.iterrows():
    person_id = row.iloc[0]  # First column = ID
    for col in session_cols:
        val = row[col]
        if pd.notna(val):
            session_file = str(val).strip().lower()
            session_file = os.path.splitext(session_file)[0]  # remove .csv
            file_to_person[session_file] = person_id

print(f"Total sessions in mapping: {len(file_to_person)}")

# ---- Group files by person ----
person_sessions = defaultdict(list)
unmatched = []

for file in os.listdir(kalman_filter_data_path):
    if not file.endswith('.csv'):
        continue
    # Strip leading/trailing spaces and remove multiple .csv extensions
    name = file.strip().lower()
    while name.endswith('.csv'):
        name = name[:-4]  # remove '.csv'

    person_id = file_to_person.get(name)
    if person_id is not None:
        person_sessions[person_id].append(file)
    else:
        unmatched.append(file)

print(f"Matched people: {len(person_sessions)}")
print(f"Unmatched files: {len(unmatched)}")
if unmatched:
    print("Examples:", unmatched[:5])

# ---- Storage ----
X_train, y_train = [], []
X_val, y_val = [], []
X_test, y_test = [], []

# ---- Helper: Interpolate and extract windows ----
def process_file(file_path, person_id):
    df = pd.read_csv(file_path)
    df['window_id'] = (df['timestamp'] - df['timestamp'].iloc[0]) // window_ms
    chunks = []
    for _, group in df.groupby('window_id'):
        if len(group) < 2:
            continue
        time_arr = group['timestamp'].values - group['timestamp'].values[0]
        new_time = np.linspace(0, time_arr[-1], interp_pts)
        try:
            x = np.interp(new_time, time_arr, group['acc_x_kf'].values)
            y = np.interp(new_time, time_arr, group['acc_y_kf'].values)
            z = np.interp(new_time, time_arr, group['acc_z_kf'].values)
            window = np.stack((x, y, z), axis=-1)  # shape: (500, 3)
            chunks.append(window)
        except Exception:
            continue
    return chunks

# ---- Split by session ----
used_people = 0

for person_id, sessions in person_sessions.items():
    if len(sessions) < 3:
        continue

    sessions = sorted(sessions)
    test_file = sessions[0]
    val_file = sessions[1]
    train_files = sessions[2:]

    used_people += 1

    for file in train_files:
        windows = process_file(os.path.join(kalman_filter_data_path, file), person_id)
        X_train.extend(windows)
        y_train.extend([person_id] * len(windows))

    for file in [val_file]:
        windows = process_file(os.path.join(kalman_filter_data_path, file), person_id)
        X_val.extend(windows)
        y_val.extend([person_id] * len(windows))

    for file in [test_file]:
        windows = process_file(os.path.join(kalman_filter_data_path, file), person_id)
        X_test.extend(windows)
        y_test.extend([person_id] * len(windows))

# ---- Save as structured npy files ----
Path(out_path).mkdir(parents=True, exist_ok=True)

dataset_X = SimpleNamespace(train=np.array(X_train), val=np.array(X_val), test=np.array(X_test))
dataset_y = SimpleNamespace(train=np.array(y_train), val=np.array(y_val), test=np.array(y_test))

np.save(f"{out_path}/dataset_X.npy", dataset_X, allow_pickle=True)
np.save(f"{out_path}/dataset_y.npy", dataset_y, allow_pickle=True)

# ---- Print summary ----
def print_split_info(name, X, y):
    print(f"\n{name} set:")
    print(f"  Samples:        {len(y)}")
    print(f"  Unique persons: {len(np.unique(y))}")

print("\n=== Dataset Split Summary ===")
print_split_info("Train", X_train, y_train)
print_split_info("Validation", X_val, y_val)
print_split_info("Test", X_test, y_test)

print(f"\nTotal persons in matrix: {matrix.shape[0]}")
print(f"Persons with ≥3 sessions used: {used_people}")


Total sessions in mapping: 338
Matched people: 92
Unmatched files: 0

=== Dataset Split Summary ===

Train set:
  Samples:        9088
  Unique persons: 62

Validation set:
  Samples:        5935
  Unique persons: 62

Test set:
  Samples:        6020
  Unique persons: 62

Total persons in matrix: 93
Persons with ≥3 sessions used: 62


In [9]:
from collections import Counter

# Count occurrences of each label
label_counts = Counter(y_train)

# Find label with the fewest entries
min_label, min_count = min(label_counts.items(), key=lambda x: x[1])

max_label, max_count = max(label_counts.items(), key=lambda x: x[1])

print(f"\nLabel with fewest samples: {min_label} ({min_count} samples)")
print(f"Label with most samples: {max_label} ({max_count} samples)")



Label with fewest samples: 74 (72 samples)
Label with most samples: 51 (270 samples)


In [10]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Flatten
from tensorflow.keras.utils import to_categorical
from sklearn.utils import shuffle


2025-05-02 15:33:26.947345: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-02 15:33:26.957693: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-02 15:33:27.044022: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-02 15:33:27.113704: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746189207.187933   26841 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746189207.21

In [16]:
data_path = '../data/kaggle-drdataboston/attempt_5'
fname = 'dataset'

X_data = np.load(f'{data_path}/{fname}_X.npy', allow_pickle=True).item()  # shape: (N, T)
y_data = np.load(f'{data_path}/{fname}_y.npy', allow_pickle=True).item()  # shape: (N,)

print(X.train.shape)
print(y.train.shape)



(9088, 500, 3)
(9088,)


In [19]:
X_train = X_data.train
X_val   = X_data.val
X_test  = X_data.test

y_train_raw = y_data.train
y_val_raw   = y_data.val
y_test_raw  = y_data.test

from sklearn.preprocessing import LabelEncoder
import joblib

# Fit on the full set of labels
label_encoder = LabelEncoder()
y_all_raw = np.concatenate([y_train_raw, y_val_raw, y_test_raw])
y_all_enc = label_encoder.fit_transform(y_all_raw)

# Save it
joblib.dump(label_encoder, f'{data_path}/label_encoder.pkl')

# Encode each split
y_train = label_encoder.transform(y_train_raw)
y_val   = label_encoder.transform(y_val_raw)
y_test  = label_encoder.transform(y_test_raw)



In [20]:
# X shape: (N, T, 3)
X_train_scaled = np.zeros_like(X_train)
X_val_scaled = np.zeros_like(X_val)
X_test_scaled = np.zeros_like(X_test)

scalers = []

for i in range(X_train.shape[2]):  # loop over channels: 0=x, 1=y, 2=z
    scaler = StandardScaler()
    scaler.fit(X_train[:, :, i])  # fit on (N, T) for this channel
    
    X_train_scaled[:, :, i] = scaler.transform(X_train[:, :, i])
    X_val_scaled[:, :, i]   = scaler.transform(X_val[:, :, i])
    X_test_scaled[:, :, i]  = scaler.transform(X_test[:, :, i])
    
    scalers.append(scaler)

# Optionally save all 3 scalers
joblib.dump(scalers, f'{data_path}/scalers.pkl')


['../data/kaggle-drdataboston/attempt_5/scalers.pkl']

In [21]:
# Now you can get num_classes safely
num_classes = len(np.unique(y_all_enc))

# One-hot encode
from tensorflow.keras.utils import to_categorical
y_train_cat = to_categorical(y_train, num_classes)
y_val_cat = to_categorical(y_val, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

In [22]:
from sklearn.utils import class_weight
import numpy as np

# Compute weights for each class
class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convert to dict for Keras
class_weights_dict = dict(enumerate(class_weights_array))


In [25]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Dropout, BatchNormalization, Dense, GlobalAveragePooling1D
from tensorflow.keras.optimizers import Adam

model3 = Sequential([
    Input(shape=(X_train_scaled.shape[1], X_train_scaled.shape[2])),  # (500, 3)

    Conv1D(64, kernel_size=5, padding='same'),
    BatchNormalization(),
    Dropout(0.1),
    MaxPooling1D(pool_size=2),
    
    Conv1D(128, kernel_size=5, padding='same'),
    BatchNormalization(),
    Dropout(0.1),
    MaxPooling1D(pool_size=2),
    
    Conv1D(256, kernel_size=5, padding='same'),
    BatchNormalization(),
    Dropout(0.1),
    MaxPooling1D(pool_size=2),
    
    GlobalAveragePooling1D(),

    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

model3.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model3.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)               │ (None, 500, 64)        │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 500, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 500, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 250, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 250, 128)       │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 250, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 250, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 125, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 125, 256)       │       164,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 125, 256)       │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 125, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 62, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 62)             │         7,998 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 248,894 (972.24 KB)

 Trainable params: 247,998 (968.74 KB)

 Non-trainable params: 896 (3.50 KB)

In [26]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


model3.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_val_scaled, y_val_cat),
    epochs=30,
    batch_size=32,
    verbose=1,
    callbacks = [early_stop],
    class_weight=class_weights_dict
)


Epoch 1/30
284/284 ━━━━━━━━━━━━━━━━━━━━ 20s 66ms/step - accuracy: 0.0732 - loss: 3.8447 - val_accuracy: 0.0283 - val_loss: 4.3611
Epoch 2/30
284/284 ━━━━━━━━━━━━━━━━━━━━ 18s 65ms/step - accuracy: 0.1794 - loss: 3.2097 - val_accuracy: 0.0298 - val_loss: 5.2408
Epoch 3/30
284/284 ━━━━━━━━━━━━━━━━━━━━ 19s 66ms/step - accuracy: 0.2298 - loss: 2.9029 - val_accuracy: 0.0239 - val_loss: 5.8299
Epoch 4/30
284/284 ━━━━━━━━━━━━━━━━━━━━ 18s 65ms/step - accuracy: 0.2807 - loss: 2.6185 - val_accuracy: 0.0241 - val_loss: 6.3480
Epoch 5/30
284/284 ━━━━━━━━━━━━━━━━━━━━ 19s 66ms/step - accuracy: 0.3338 - loss: 2.3831 - val_accuracy: 0.0243 - val_loss: 6.8407
Epoch 6/30
284/284 ━━━━━━━━━━━━━━━━━━━━ 19s 66ms/step - accuracy: 0.3886 - loss: 2.1697 - val_accuracy: 0.0281 - val_loss: 7.2925


In [17]:
from tensorflow.keras.models import load_model

model_lstm = load_model(f'{data_path}/CNN_LSTM/best_cnn_lstm.keras')

In [18]:
test_loss, test_acc = model_lstm.evaluate(X_test_scaled, y_test_cat)
print(f"Test accuracy: {test_acc:.2f}")


123/123 ━━━━━━━━━━━━━━━━━━━━ 7s 52ms/step - accuracy: 0.9607 - loss: 0.1744
Test accuracy: 0.96


In [16]:
test_loss, test_acc = model3.evaluate(X_test_scaled, y_test_cat)
print(f"Test accuracy: {test_acc:.2f}")

model3.save('../data/kaggle-drdataboston/attempt_3/model3-cnn.keras')


123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8684 - loss: 0.5820
Test accuracy: 0.87
